In [1]:
!nvidia-smi

Sat May 23 09:22:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile dna_count.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void countDNA(const char *dna, int n, int *counts) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        switch (dna[i]) {
            case 'A':
                atomicAdd(&counts[0], 1);
                break;
            case 'C':
                atomicAdd(&counts[1], 1);
                break;
            case 'G':
                atomicAdd(&counts[2], 1);
                break;
            case 'T':
                atomicAdd(&counts[3], 1);
                break;
        }
    }
}

int main() {
    const char *dna = "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC";
    int n = strlen(dna);

    int h_counts[4] = {0};

    char *d_dna;
    int *d_counts;

    cudaMalloc(&d_dna, n * sizeof(char));
    cudaMalloc(&d_counts, 4 * sizeof(int));

    cudaMemcpy(d_dna, dna, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemset(d_counts, 0, 4 * sizeof(int));

    int blockSize = 128;
    int gridSize = (n + blockSize - 1) / blockSize;

    countDNA<<<gridSize, blockSize>>>(d_dna, n, d_counts);
    cudaDeviceSynchronize();

    cudaMemcpy(h_counts, d_counts, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("%d %d %d %d\n", h_counts[0], h_counts[1], h_counts[2], h_counts[3]);

    cudaFree(d_dna);
    cudaFree(d_counts);

    return 0;
}

Writing dna_count.cu


In [11]:
!nvcc -Wno-deprecated-gpu-targets dna_count.cu -o dna_count

In [12]:
!./dna_count

20 12 17 21


In [13]:
%%writefile dna_to_rna.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void transcribeDNA(char *d_str, int length) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < length && d_str[idx] == 'T') {
        d_str[idx] = 'U';
    }
}

int main() {
    char h_str[] = "GATGGAACTTGACTACGTAAATT";
    int length = strlen(h_str);

    char *d_str;

    cudaMalloc((void**)&d_str, (length + 1) * sizeof(char));
    cudaMemcpy(d_str, h_str, (length + 1) * sizeof(char), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (length + threadsPerBlock - 1) / threadsPerBlock;

    transcribeDNA<<<blocksPerGrid, threadsPerBlock>>>(d_str, length);
    cudaDeviceSynchronize();

    cudaMemcpy(h_str, d_str, (length + 1) * sizeof(char), cudaMemcpyDeviceToHost);

    printf("%s\n", h_str);

    cudaFree(d_str);

    return 0;
}

Writing dna_to_rna.cu


In [14]:
!nvcc -Wno-deprecated-gpu-targets dna_to_rna.cu -o dna_to_rna

In [15]:
!./dna_to_rna

GAUGGAACUUGACUACGUAAAUU


In [19]:
%%writefile protein_weight.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__device__ float mass(char aa) {
    switch (aa) {
        case 'A': return 71.03711f;
        case 'C': return 103.00919f;
        case 'D': return 115.02694f;
        case 'E': return 129.04259f;
        case 'F': return 147.06841f;
        case 'G': return 57.02146f;
        case 'H': return 137.05891f;
        case 'I': return 113.08406f;
        case 'K': return 128.09496f;
        case 'L': return 113.08406f;
        case 'M': return 131.04049f;
        case 'N': return 114.04293f;
        case 'P': return 97.05276f;
        case 'Q': return 128.05858f;
        case 'R': return 156.10111f;
        case 'S': return 87.03203f;
        case 'T': return 101.04768f;
        case 'V': return 99.06841f;
        case 'W': return 186.07931f;
        case 'Y': return 163.06333f;
        default: return 0.0f;
    }
}

__global__ void proteinMass(const char *protein, int n, float *total) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        atomicAdd(total, mass(protein[i]));
    }
}

int main() {
    const char *protein = "SKADYEK";
    int n = strlen(protein);

    float result = 0.0f;

    char *d_protein;
    float *d_result;

    cudaMalloc((void**)&d_protein, n * sizeof(char));
    cudaMalloc((void**)&d_result, sizeof(float));

    cudaMemcpy(d_protein, protein, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemset(d_result, 0, sizeof(float));

    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    proteinMass<<<gridSize, blockSize>>>(d_protein, n, d_result);
    cudaDeviceSynchronize();

    cudaMemcpy(&result, d_result, sizeof(float), cudaMemcpyDeviceToHost);

    printf("%.3f\n", result);

    cudaFree(d_protein);
    cudaFree(d_result);

    return 0;
}

Overwriting protein_weight.cu


In [20]:
!nvcc -Wno-deprecated-gpu-targets protein_weight.cu -o protein_weight

In [21]:
!./protein_weight

821.392


In [22]:
%%writefile genetic_code.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__device__ char get_amino(char a, char b, char c) {

    if (a=='U' && b=='U' && (c=='U' || c=='C')) return 'F';
    if (a=='U' && b=='U' && (c=='A' || c=='G')) return 'L';

    if (a=='C' && b=='U') return 'L';

    if (a=='A' && b=='U' && c=='G') return 'M';

    if (a=='G' && b=='U') return 'V';

    if (a=='U' && b=='C') return 'S';

    if (a=='C' && b=='C') return 'P';

    if (a=='A' && b=='C') return 'T';

    if (a=='G' && b=='C') return 'A';

    if (a=='U' && b=='A' && (c=='U' || c=='C')) return 'Y';

    if ((a=='U' && b=='A' && c=='A') ||
        (a=='U' && b=='A' && c=='G') ||
        (a=='U' && b=='G' && c=='A')) return '*';

    if (a=='C' && b=='A' && (c=='U' || c=='C')) return 'H';
    if (a=='C' && b=='A' && (c=='A' || c=='G')) return 'Q';

    if (a=='A' && b=='A' && (c=='U' || c=='C')) return 'N';
    if (a=='A' && b=='A' && (c=='A' || c=='G')) return 'K';

    if (a=='G' && b=='A' && (c=='U' || c=='C')) return 'D';
    if (a=='G' && b=='A' && (c=='A' || c=='G')) return 'E';

    if (a=='U' && b=='G' && (c=='U' || c=='C')) return 'C';
    if (a=='U' && b=='G' && c=='G') return 'W';

    if (a=='C' && b=='G') return 'R';
    if (a=='A' && b=='G' && (c=='U' || c=='C')) return 'S';
    if (a=='A' && b=='G' && (c=='A' || c=='G')) return 'R';

    if (a=='G' && b=='G') return 'G';

    if (a=='A' && b=='U' && (c=='U' || c=='C' || c=='A')) return 'I';

    return '?';
}

__global__ void translateRNA(char *rna, char *protein, int codons) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < codons) {

        int pos = idx * 3;

        char amino = get_amino(rna[pos], rna[pos+1], rna[pos+2]);

        if (amino == '*') {
            protein[idx] = '\0';
        } else {
            protein[idx] = amino;
        }
    }
}

int main() {

    char h_rna[] = "AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGA";

    int length = strlen(h_rna);
    int codons = length / 3;

    char h_protein[100] = {0};

    char *d_rna;
    char *d_protein;

    cudaMalloc((void**)&d_rna, length * sizeof(char));
    cudaMalloc((void**)&d_protein, codons * sizeof(char));

    cudaMemcpy(d_rna, h_rna, length * sizeof(char), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (codons + threads - 1) / threads;

    translateRNA<<<blocks, threads>>>(d_rna, d_protein, codons);

    cudaDeviceSynchronize();

    cudaMemcpy(h_protein, d_protein, codons * sizeof(char), cudaMemcpyDeviceToHost);

    printf("%s\n", h_protein);

    cudaFree(d_rna);
    cudaFree(d_protein);

    return 0;
}

Writing genetic_code.cu


In [23]:
!nvcc -Wno-deprecated-gpu-targets genetic_code.cu -o genetic_code

In [24]:
!./genetic_code

MAIVMGR


In [25]:
%%writefile hamming_distance.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <string.h>

__global__ void hammingDistance(const char *s, const char *t, int n, int *count) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n && s[i] != t[i]) {
        atomicAdd(count, 1);
    }
}

int main() {
    const char *s = "GAGCCTACTAACGGGAT";
    const char *t = "CATCGTAATGACGGCCT";

    int n = strlen(s);
    int result = 0;

    char *d_s, *d_t;
    int *d_count;

    cudaMalloc((void**)&d_s, n * sizeof(char));
    cudaMalloc((void**)&d_t, n * sizeof(char));
    cudaMalloc((void**)&d_count, sizeof(int));

    cudaMemcpy(d_s, s, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemcpy(d_t, t, n * sizeof(char), cudaMemcpyHostToDevice);
    cudaMemset(d_count, 0, sizeof(int));

    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    hammingDistance<<<gridSize, blockSize>>>(d_s, d_t, n, d_count);
    cudaDeviceSynchronize();

    cudaMemcpy(&result, d_count, sizeof(int), cudaMemcpyDeviceToHost);

    printf("%d\n", result);

    cudaFree(d_s);
    cudaFree(d_t);
    cudaFree(d_count);

    return 0;
}

Writing hamming_distance.cu


In [26]:
!nvcc -Wno-deprecated-gpu-targets hamming_distance.cu -o hamming_distance

In [27]:
!./hamming_distance

7
